# Adaptive Stop Loss via ATR on SPY
## Strategy Brief
This strategy uses the Average True Range (ATR) to set adaptive stop losses for trades on SPY. The ATR provides a measure of volatility, allowing for dynamic adjustment of stop loss levels to accommodate changing market conditions. The strategy enters trades based on a simple moving average crossover and exits using the adaptive stop loss. The goal is to improve risk management and potentially enhance returns compared to a static stop loss approach.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for our strategy, including the moving average period, ATR period, and stop loss multiplier.

In [ ]:
MA_PERIOD = 50
ATR_PERIOD = 14
STOP_LOSS_MULTIPLIER = 1.5
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'

## PHASE 2 - Data Exploration
We will download historical data for SPY from Yahoo Finance, compute the ATR, and plot it alongside the price to understand the volatility context.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Compute ATR
high_low = data['High'] - data['Low']
high_close = np.abs(data['High'] - data['Close'].shift())
low_close = np.abs(data['Low'] - data['Close'].shift())
tr = high_low.combine(high_close, max).combine(low_close, max)
data['ATR'] = tr.rolling(ATR_PERIOD).mean()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(data['ATR'], label='ATR', linestyle='--')
plt.title('SPY Price and ATR')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We will create a signal based on a simple moving average crossover and implement the adaptive stop loss using the ATR.

In [ ]:
# Compute moving average
data['MA'] = data['Close'].rolling(MA_PERIOD).mean()

# Signal: 1 for buy, -1 for sell
data['Signal'] = 0
data.loc[data['Close'] > data['MA'], 'Signal'] = 1
data.loc[data['Close'] < data['MA'], 'Signal'] = -1

# Adaptive Stop Loss
initial_stop_loss = data['Close'] - (STOP_LOSS_MULTIPLIER * data['ATR'])
data['Stop_Loss'] = initial_stop_loss

# Positions
data['Position'] = data['Signal']

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns and plotting the equity curve.

In [ ]:
# Shift positions to align with next day's returns
data['Position'] = data['Position'].shift(1)

# Calculate daily returns
data['Market_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = data['Market_Return'] * data['Position']

# Calculate equity curve
data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.title('Equity Curve of the Strategy')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown, and compare it with a buy-and-hold approach.

In [ ]:
def calculate_metrics(data):
    cagr = (data['Equity_Curve'].iloc[-1] ** (252.0 / len(data))) - 1
    sharpe = data['Strategy_Return'].mean() / data['Strategy_Return'].std() * np.sqrt(252)
    downside_std = data[data['Strategy_Return'] < 0]['Strategy_Return'].std()
    sortino = data['Strategy_Return'].mean() / downside_std * np.sqrt(252)
    max_drawdown = ((data['Equity_Curve'].cummax() - data['Equity_Curve']).max()) / data['Equity_Curve'].cummax().max()
    calmar = cagr / max_drawdown
    return cagr, sharpe, sortino, calmar, max_drawdown

strategy_metrics = calculate_metrics(data)

# Buy and Hold
bh_cagr = (data['Close'].iloc[-1] / data['Close'].iloc[0]) ** (252.0 / len(data)) - 1
bh_max_drawdown = ((data['Close'].cummax() - data['Close']).max()) / data['Close'].cummax().max()

# Display results
comparison = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe', 'Sortino', 'Calmar', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy & Hold': [bh_cagr, np.nan, np.nan, np.nan, bh_max_drawdown]
})
print(comparison)

## PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of data, compute today's signal, and print the current position.

In [ ]:
def get_latest_signal():
    recent_data = yf.download('SPY', period='60d')
    recent_data['MA'] = recent_data['Close'].rolling(MA_PERIOD).mean()
    recent_data['ATR'] = tr.rolling(ATR_PERIOD).mean()
    recent_data['Signal'] = 0
    recent_data.loc[recent_data['Close'] > recent_data['MA'], 'Signal'] = 1
    recent_data.loc[recent_data['Close'] < recent_data['MA'], 'Signal'] = -1
    current_signal = recent_data['Signal'].iloc[-1]
    print(f"Current Position: {'Long' if current_signal == 1 else 'Short' if current_signal == -1 else 'Neutral'}")

get_latest_signal()